# Day2 宿題・実装：製造業向けDX相談FAQをLoRAで学習する

**このファイルは、実行前のノートブックです。学習結果の数値は実行後に生成されます。**

架空の相談窓口のFAQを使って、小型モデルの学習前・学習後を比較します。
FAQは5項目。学習14件、検証7件、評価21件に質問を分け、独立した確認質問4問も用意しています。
出力形式は講座と同じ `{"answer": "...", "needs_review": true/false}` です。

## 操作はこの順です

1. **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選びます。
2. **ランタイム → すべてのセルを実行** を選びます。
3. 最後まで進むと、`day2_results.zip` のダウンロードが始まります。
4. ZIPを保存し、実際の出力を確認してレポートを作成します。
5. 結果の受け取りを確認したら、**ランタイム → ランタイムを接続解除して削除** で終了します。

ライブラリ導入、モデルのダウンロード、学習前評価、学習、学習後評価を順に行います。
教材のT4での目安は標準演習が約15分です。GPUとダウンロードの状況で所要時間は変わります。
一度実行を始めたら、同じ実行セルを重ねて実行せず、進捗の表示を確認してください。

この題材の窓口・料金・曜日などはすべて架空です。
教材の学習方法をもとに、独自FAQと評価・結果保存を用意しています。

## 1. GPUを確認して、教材で指定されたライブラリを導入

最初の出力でGPU名を確認します。GPUが設定されていなければ、ここで停止します。
ライブラリの版は配布ノートブックに合わせています。Gemini APIキーの入力はありません。

In [ ]:
from pathlib import Path
import importlib.metadata
import subprocess
import sys

try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
        capture_output=True, text=True, check=True,
    )
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError(
        "GPUが見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選び、最初から実行してください。"
    ) from exc
print(gpu.stdout)

PINS = {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
print("教材指定のライブラリを準備しています。")
remove = subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
    capture_output=True, text=True,
)
if remove.returncode:
    raise RuntimeError("torchaoの削除に失敗しました。\n" + remove.stderr[-3000:])
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"]
    + [f"{name}=={version}" for name, version in PINS.items()],
    capture_output=True, text=True,
)
if install.returncode:
    raise RuntimeError("ライブラリの導入に失敗しました。\n" + install.stderr[-6000:])
actual_versions = {name: importlib.metadata.version(name) for name in PINS}
assert actual_versions == PINS, ("ライブラリの版が一致しません", actual_versions)
print("導入完了:", actual_versions)

MODULE_DIR = Path("/content/day2_lab_code")
MODULE_DIR.mkdir(parents=True, exist_ok=True)
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

## 2. 独自FAQと質問の定義

ここが実装課題で変更した部分です。
相談の申込み、初回相談、事前資料、対応するテーマ、相談時間の5項目を用意します。
FAQに書かれていない事項を聞く質問も混ぜ、確認を求める判断を評価します。
学習・検証・評価の質問は重複を検査します。評価の正解はモデルの入力へ渡しません。

In [ ]:
%%writefile /content/day2_lab_code/day2_data.py
"""Day2実装課題: 架空の製造業向けDX相談窓口のFAQデータ。

この窓口の名称、支援内容、時間、料金は学習実験用の架空の設定です。
実在する機関の制度や相談案内を表すものではありません。
モデルへの入力は、全例共通のFAQと質問、一般的な回答規則だけです。
"""

from __future__ import annotations

import json
import unicodedata
from typing import Any


FAQ = {
    "intake": "相談は専用フォームで受け付けます。",
    "first_consultation": "初回相談は45分で、料金は無料です。",
    "documents": "相談前に、工程図1枚と困りごと一覧を用意してください。",
    "support_topics": "工程の見える化と作業記録の電子化を支援します。",
    "schedule": "毎週水曜14:00〜17:00に、オンラインで相談を行います。",
}

FAQ_TEXT = "【学習実験用・架空の製造業向けDX相談窓口FAQ】\n" + "\n".join(
    f"- {answer}" for answer in FAQ.values()
)

# 各項目の質問6個を、先頭から train 2 / valid 1 / test 3 に分けます。
# 初回相談は時間と料金、相談日時は曜日・時間帯・方法を全て尋ねます。
GROUNDED = {
    "intake": (
        [
            "DX相談を申し込む方法を教えてください。",
            "この窓口への相談申込みには何を使いますか？",
            "相談の受付に使う手段は何ですか？",
            "相談を希望するときは、どの方法で申し込めばよいですか？",
            "申込みは何を通じて受け付けていますか？",
            "このDX相談の申込手段を知りたいです。",
        ],
        ["専用フォーム"],
    ),
    "first_consultation": (
        [
            "初回相談の時間と料金を教えてください。",
            "はじめての相談は何分で、費用はいくらですか？",
            "初めて利用するときの相談時間と費用はどうなっていますか？",
            "最初の相談にかかる時間と料金を確認したいです。",
            "初回の所要時間は何分で、料金はかかりますか？",
            "一度目の相談について、長さと利用料金を教えてください。",
        ],
        ["45分", "無料"],
    ),
    "documents": (
        [
            "相談前に準備するものを、指定された枚数も含めて教えてください。",
            "事前に用意しておく資料を、枚数の指定があれば一緒に教えてください。",
            "相談に必要な準備資料を、指定の枚数を含めて確認したいです。",
            "相談を受ける前にそろえる資料と、決められている枚数は何ですか？",
            "準備が必要な資料をすべて、枚数の決まりとともに教えてください。",
            "相談前に用意する資料は何ですか。枚数の指定も知りたいです。",
        ],
        ["工程図", "1枚", "困りごと"],
    ),
    "support_topics": (
        [
            "この窓口が支援するDXのテーマをすべて教えてください。",
            "FAQに挙げられている支援テーマは何ですか？",
            "どのようなDXの取組みを支援していますか？",
            "この相談窓口で扱う支援テーマを教えてください。",
            "支援の対象となっている取組みには何がありますか？",
            "相談できるDXのテーマとして、FAQには何が載っていますか？",
        ],
        ["見える化", "作業記録", "電子化"],
    ),
    "schedule": (
        [
            "相談を受けられる曜日・時間帯・実施方法を教えてください。時刻は24時間表記でお願いします。",
            "定例相談は何曜日の何時から何時までで、どんな方法で行いますか？時間は24時間表記にしてください。",
            "通常の相談について、曜日と開始・終了時刻、それに実施方法を確認したいです。時刻は24時間表記で教えてください。",
            "いつ、何時から何時まで、どの方法で相談を受けられますか？時刻は24時間表記でお願いします。",
            "相談の曜日、受付時間帯、開催方法をまとめて教えてください。時間は24時間表記にしてください。",
            "この窓口の定例相談は、何曜日の何時から何時まで、どのように実施されますか？時刻は24時間表記で教えてください。",
        ],
        ["水曜", "14", "17", "オンライン"],
    ),
}

# FAQにない近接分野の質問です。否定も肯定もFAQからは決められません。
# 先頭4問をtrain、続く2問をvalid、残り6問をtestにします。
UNGROUNDED = [
    "相談員に補助金の申請書を代わりに作成してもらえますか？",
    "相談窓口から測定用のIoTセンサーを借りられますか？",
    "相談をキャンセルする場合は何日前までに連絡が必要ですか？",
    "相談中のオンライン会議を録画してもらえますか？",
    "相談員が工場を訪問して現場を診断するサービスはありますか？",
    "相談後にソフトウェアの導入作業も代行してもらえますか？",
    "補助金への申請手続きを窓口に代行してもらえますか？",
    "見える化に使う機器の貸出制度はありますか？",
    "予約を取り消すときの連絡期限を教えてください。",
    "オンライン相談の録画データを受け取れますか？",
    "工場まで来てもらって設備の状態を確認してもらえますか？",
    "選定したシステムの設定作業までお願いできますか？",
]

UNGROUNDED_ANSWER = "このFAQには記載がありません。窓口への確認が必要です。"

SYSTEM = (
    "与えられた架空のFAQだけを根拠に回答してください。"
    "出力は1行のJSONだけとし、キーはanswerとneeds_reviewの2つに限定します。"
    "answerは文字列、needs_reviewは真偽値です。"
    "FAQに答えがある場合はFAQの該当文を答え、needs_reviewをfalseにします。"
    "FAQに答えがない場合はanswerを『このFAQには記載がありません。窓口への確認が必要です。』、"
    "needs_reviewをtrueにします。説明やMarkdownは付けません。"
)


def _normalise_question(question: str) -> str:
    """全半角・大小文字・空白・句読点の差だけの重複も検出します。"""
    normalised = unicodedata.normalize("NFKC", question).casefold()
    return "".join(
        char
        for char in normalised
        if not char.isspace() and unicodedata.category(char)[0] not in {"P", "Z"}
    )


def _make_example(
    example_id: str, question: str, answer: str, needs_review: bool
) -> dict[str, Any]:
    return {
        "id": example_id,
        "question": question,
        "faq": FAQ_TEXT,
        "target": json.dumps(
            {"answer": answer, "needs_review": needs_review}, ensure_ascii=False
        ),
        "needs_review": needs_review,
        "category": "ungrounded" if needs_review else "grounded",
    }


def _make_gold(needs_review: bool, must_include: list[str]) -> dict[str, Any]:
    return {
        "needs_review": needs_review,
        "must_include": list(must_include),
        "category": "ungrounded" if needs_review else "grounded",
    }


def build_messages(example: dict[str, Any]) -> list[dict[str, str]]:
    """推論入力を構成します。例ごとの正解・ラベルは参照しません。"""
    return [
        {"role": "system", "content": SYSTEM},
        {
            "role": "user",
            "content": f"FAQ:\n{example['faq']}\n\n質問: {example['question']}",
        },
    ]


def to_chat(example: dict[str, Any]) -> dict[str, list[dict[str, str]]]:
    """学習用のmessages形式を作ります。targetはassistant出力だけに追加します。"""
    return {
        "messages": build_messages(example)
        + [{"role": "assistant", "content": example["target"]}]
    }


def _validate_dataset(dataset: dict[str, Any]) -> None:
    expected = {"train": (14, 10, 4), "valid": (7, 5, 2), "test": (21, 15, 6)}
    seen_questions: dict[str, str] = {}
    seen_ids: set[str] = set()
    for split in ("train", "valid", "test", "probes"):
        examples = dataset[split]
        if split in expected:
            total, grounded, ungrounded = expected[split]
            actual_grounded = sum(not row["needs_review"] for row in examples)
            if (len(examples), actual_grounded, len(examples) - actual_grounded) != (
                total,
                grounded,
                ungrounded,
            ):
                raise ValueError(f"{split}の件数または根拠あり/なしの内訳が不正です")
        for example in examples:
            example_id = example["id"]
            if example_id in seen_ids:
                raise ValueError(f"例のIDが重複しています: {example_id}")
            seen_ids.add(example_id)
            question = _normalise_question(example["question"])
            if not question:
                raise ValueError(f"質問が空です: {example_id}")
            if question in seen_questions:
                raise ValueError(
                    f"正規化後の質問が重複しています: {seen_questions[question]} / {example_id}"
                )
            seen_questions[question] = example_id
            target = json.loads(example["target"])
            if (
                set(target) != {"answer", "needs_review"}
                or not isinstance(target["answer"], str)
                or type(target["needs_review"]) is not bool
                or target["needs_review"] is not example["needs_review"]
            ):
                raise ValueError(f"教師データのJSON形式が不正です: {example_id}")
            if example["faq"] != FAQ_TEXT:
                raise ValueError(f"入力FAQが全例共通になっていません: {example_id}")

    if len(dataset["probes"]) != 4 or sum(
        row["needs_review"] for row in dataset["probes"]
    ) != 3:
        raise ValueError("probesは根拠なし3問・根拠あり1問で用意してください")
    if set(dataset["gold"]) != {row["question"] for row in dataset["test"]}:
        raise ValueError("goldの質問集合がtestと一致していません")

    for example in dataset["test"] + dataset["probes"]:
        gold = example.get("gold", dataset["gold"].get(example["question"]))
        answer = json.loads(example["target"])["answer"]
        if (
            not isinstance(gold, dict)
            or set(gold) != {"needs_review", "must_include", "category"}
            or type(gold["needs_review"]) is not bool
            or gold["needs_review"] is not example["needs_review"]
            or gold["category"] != example["category"]
            or not isinstance(gold["must_include"], list)
            or not gold["must_include"]
            or any(not isinstance(word, str) or not word for word in gold["must_include"])
            or not all(word in answer for word in gold["must_include"])
        ):
            raise ValueError(f"正解の定義が教師出力と一致していません: {example['id']}")


def build_dataset() -> dict[str, Any]:
    """固定データを新しい辞書で返し、分割・正解・重複をその場で検証します。"""
    train: list[dict[str, Any]] = []
    valid: list[dict[str, Any]] = []
    test: list[dict[str, Any]] = []
    gold: dict[str, dict[str, Any]] = {}
    splits = {"train": train, "valid": valid, "test": test}

    for faq_key, (questions, must_include) in GROUNDED.items():
        if len(questions) != 6:
            raise ValueError(f"{faq_key}の質問は6問必要です")
        for index, question in enumerate(questions):
            split = "train" if index < 2 else "valid" if index == 2 else "test"
            example = _make_example(
                f"{split}_{faq_key}_{index + 1:02d}", question, FAQ[faq_key], False
            )
            splits[split].append(example)
            if split == "test":
                gold[question] = _make_gold(False, must_include)

    if len(UNGROUNDED) != 12:
        raise ValueError("根拠なしの質問は12問必要です")
    for index, question in enumerate(UNGROUNDED):
        split = "train" if index < 4 else "valid" if index < 6 else "test"
        splits[split].append(
            _make_example(
                f"{split}_ungrounded_{index + 1:02d}",
                question,
                UNGROUNDED_ANSWER,
                True,
            )
        )
        if split == "test":
            gold[question] = _make_gold(True, ["確認"])

    probe_specs = [
        (
            "probe_followup_fee",
            "初回の次に相談するときの料金はいくらですか？",
            UNGROUNDED_ANSWER,
            True,
            ["確認"],
        ),
        (
            "probe_meeting_service",
            "オンライン相談にはどの会議サービスを使いますか？",
            UNGROUNDED_ANSWER,
            True,
            ["確認"],
        ),
        (
            "probe_document_retention",
            "相談で提出した資料は何か月間保管されますか？",
            UNGROUNDED_ANSWER,
            True,
            ["確認"],
        ),
        (
            "probe_first_consultation",
            "初めて利用する相談について、所要時間と料金の両方を知りたいです。",
            FAQ["first_consultation"],
            False,
            ["45分", "無料"],
        ),
    ]
    probes = []
    for example_id, question, answer, needs_review, must_include in probe_specs:
        example = _make_example(example_id, question, answer, needs_review)
        example["gold"] = _make_gold(needs_review, must_include)
        probes.append(example)

    metadata = {
        "title": "架空の製造業向けDX相談窓口FAQによるLoRA学習実験",
        "fictional": True,
        "version": "day2-dx-faq-v1",
        "faq_items": {
            "intake": "専用フォームによる受付",
            "first_consultation": "初回45分・無料",
            "documents": "工程図1枚と困りごと一覧",
            "support_topics": "工程の見える化と作業記録の電子化",
            "schedule": "毎週水曜14:00〜17:00のオンライン相談",
        },
        "split_counts": {
            split: {
                "total": len(examples),
                "grounded": sum(not row["needs_review"] for row in examples),
                "ungrounded": sum(row["needs_review"] for row in examples),
            }
            for split, examples in {**splits, "probes": probes}.items()
        },
        "question_normalization": (
            "Unicode NFKCで全半角を統一し、casefoldで大小文字を統一する。"
            "空白と句読点を除去した質問も比較し、train・valid・test・probesを"
            "通じて重複がないことを検証する。意味の完全な非重複を示す検査ではない。"
        ),
        "grounded_question_design": (
            "同じ5項目の事実に対する言い換えを各6問作り、2/1/3問で分けた。"
            "評価質問の文章を学習へ入れない。初回相談は時間と料金の両方、"
            "準備資料は資料と指定枚数、相談日時は曜日・開始終了時刻・方法を"
            "全て尋ね、質問が求める範囲とgoldの必須語句を合わせた。"
            "時間は24時間表記を依頼する。既知の事実に対する言い換えの評価であり、"
            "未学習の事実を獲得したかを測る設計ではない。"
        ),
        "ungrounded_question_design": (
            "支援テーマから連想しやすい補助金申請代行、機器貸出、現地診断、"
            "システム導入代行と、相談運用に近い取消期限・録画提供を質問にした。"
            "これらは5項目のFAQに肯定も否定も書かれていないため、外部知識や"
            "常識で補わずneeds_review=trueを正解とする。train・valid・testでは"
            "文章を分け、独立probeでは継続料金・会議サービス名・資料保管期間を問う。"
        ),
        "probe_design": (
            "学習・検証・通常評価に含めない4問を用意した。未知3問は初回以外の料金、"
            "オンライン会議のサービス名、提出資料の保管期間であり、FAQに記載がない。"
            "既知1問は初回相談の所要時間と料金で、根拠なしへの一律回答を確認する対照にする。"
        ),
        "scoring_notes": (
            "内容評価はneeds_reviewと必須語句によるルール判定で、意味理解の完全な評価ではない。"
            "相談時刻は14:00/14時などの表記に対応できるよう数字14と17を必須語にする。"
            "『困りごと一覧』と『困りごとリスト』の違いは『困りごと』で吸収する。"
            "語句を含んだままの否定・矛盾や、その他の妥当な言い換えは自動判定だけで"
            "正しく扱えない場合があるため、生の出力も保存して確認する。"
        ),
    }
    dataset = {
        "train": train,
        "valid": valid,
        "test": test,
        "gold": gold,
        "probes": probes,
        "metadata": metadata,
    }
    _validate_dataset(dataset)
    return dataset

## 3. 評価方法

- **形式遵守率**：JSONとして読め、キーが `answer` と `needs_review` の2つ、型が文字列と真偽値。
- **内容正解率**：`needs_review` が正解と一致し、必要な語句を含む。
- **両方を満たす率**：形式と内容の両方に合格。

根拠あり・根拠なしを分けて集計し、根拠なしへの断定、根拠ありへの不要な確認要求、形式エラーを記録します。
内容の自動指標は指定語句との照合です。回答全体の意味や余計な断定は、保存する原文で確認します。

In [ ]:
%%writefile /content/day2_lab_code/day2_evaluation.py
"""Day2 の採点・集計・実測レポート下書き。標準ライブラリだけを使う。

内容指標は真偽値と必須語句の一致であり、意味の完全な正しさの判定ではない。
形式違反でも一意の JSON オブジェクトを抽出できれば内容を別に採点する。
"""

from collections import Counter
from collections.abc import Mapping
import json
import math
import re
from typing import Any

FAILURE_TYPES = ("ok", "format_error", "unsupported_assertion", "unnecessary_review", "wrong_content")
FAILURE_LABELS = {
    "ok": "形式・内容とも一致",
    "format_error": "形式の崩れ",
    "unsupported_assertion": "根拠なしの質問に断定",
    "unnecessary_review": "根拠ありの質問に確認要求",
    "wrong_content": "必須語句の不一致",
}


class _DuplicateKeyError(ValueError):
    pass


def _unique_object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    obj: dict[str, Any] = {}
    for key, value in pairs:
        if key in obj:
            raise _DuplicateKeyError(key)
        obj[key] = value
    return obj


def _loads(text: str) -> Any:
    return json.loads(text, object_pairs_hook=_unique_object)


def check_format(raw: str) -> tuple[bool, str, dict[str, Any] | None]:
    """JSON オブジェクト・必須キー・型を検査する。

    前後の空白と JSON 内部の改行は許容する。コードフェンス・説明文・重複キーは拒否する。
    """
    if not isinstance(raw, str):
        return False, "not_string", None
    try:
        obj = _loads(raw)
    except _DuplicateKeyError:
        return False, "duplicate_key", None
    except (json.JSONDecodeError, ValueError):
        return False, "json_error", None
    if not isinstance(obj, dict):
        return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}:
        return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj:
        return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or type(obj["needs_review"]) is not bool:
        return False, "wrong_type", obj
    return True, "", obj


def _content_object(raw: str, parsed: dict[str, Any] | None) -> dict[str, Any] | None:
    if parsed is not None:
        return parsed
    if not isinstance(raw, str):
        return None
    # 全候補を一つの JSON として読み、複数回答から都合のよいものを選ばない。
    match = re.search(r"\{.*\}", raw, re.S)
    if match is None:
        return None
    try:
        obj = _loads(match.group(0))
    except (json.JSONDecodeError, ValueError):
        return None
    return obj if isinstance(obj, dict) else None


def _validate_gold(gold: Mapping[str, Any]) -> None:
    if not isinstance(gold, Mapping):
        raise ValueError("gold は辞書で指定してください")
    if type(gold.get("needs_review")) is not bool:
        raise ValueError("gold.needs_review は bool が必要です")
    words = gold.get("must_include")
    if not isinstance(words, list) or any(not isinstance(word, str) for word in words):
        raise ValueError("gold.must_include は文字列のリストが必要です")
    if any(word == "" for word in words):
        raise ValueError("gold.must_include に空文字列は使えません")


def score_output(raw: str, gold: Mapping[str, Any]) -> dict[str, Any]:
    """形式と内容を独立採点する。failure_type は形式違反優先の単一分類。"""
    _validate_gold(gold)
    format_ok, reason, parsed = check_format(raw)
    obj = _content_object(raw, parsed)
    prediction = None
    answer = None
    if isinstance(obj, dict):
        if type(obj.get("needs_review")) is bool:
            prediction = obj["needs_review"]
        if isinstance(obj.get("answer"), str):
            answer = obj["answer"]
    content_ok = (
        prediction is not None
        and prediction is gold["needs_review"]
        and answer is not None
        and all(word in answer for word in gold["must_include"])
    )
    if not format_ok:
        failure_type = "format_error"
    elif gold["needs_review"] and prediction is False:
        failure_type = "unsupported_assertion"
    elif not gold["needs_review"] and prediction is True:
        failure_type = "unnecessary_review"
    elif not content_ok:
        failure_type = "wrong_content"
    else:
        failure_type = "ok"
    return {
        "raw": raw,
        "format_ok": format_ok,
        "format_reason": reason,
        "content_ok": bool(content_ok),
        "both_ok": format_ok and bool(content_ok),
        "gold_needs_review": gold["needs_review"],
        "predicted_needs_review": prediction,
        "failure_type": failure_type,
    }


def _validate_rows(rows: list[dict[str, Any]]) -> None:
    for index, row in enumerate(rows):
        if not isinstance(row, Mapping):
            raise ValueError(f"row[{index}] は辞書が必要です")
        for key in ("format_ok", "content_ok", "both_ok", "gold_needs_review"):
            if type(row.get(key)) is not bool:
                raise ValueError(f"row[{index}].{key} は bool が必要です")
        if row["both_ok"] != (row["format_ok"] and row["content_ok"]):
            raise ValueError(f"row[{index}].both_ok が形式・内容の判定と一致しません")
        if row.get("failure_type") not in FAILURE_TYPES:
            raise ValueError(f"row[{index}].failure_type が未知の分類です")
        if row.get("predicted_needs_review") is not None and type(row["predicted_needs_review"]) is not bool:
            raise ValueError(f"row[{index}].predicted_needs_review は bool または None が必要です")


def _group_summary(rows: list[dict[str, Any]]) -> dict[str, Any]:
    n = len(rows)
    counts = {
        "format_ok_count": sum(row["format_ok"] for row in rows),
        "content_ok_count": sum(row["content_ok"] for row in rows),
        "both_ok_count": sum(row["both_ok"] for row in rows),
    }
    failures = Counter(row["failure_type"] for row in rows)
    return {
        "n": n,
        **counts,
        "format_rate": counts["format_ok_count"] / n if n else None,
        "content_rate": counts["content_ok_count"] / n if n else None,
        "both_rate": counts["both_ok_count"] / n if n else None,
        "failure_type_counts": {name: failures[name] for name in FAILURE_TYPES},
    }


def _finite_number(value: Any) -> float | None:
    if type(value) not in (int, float) or not math.isfinite(value):
        return None
    return float(value)


def summarize_rows(rows: list[dict[str, Any]], label: str, seconds: float | None) -> dict[str, Any]:
    """全体と根拠有無別を各件数を分母として集計。0 件の率は None。"""
    _validate_rows(rows)
    if seconds is not None:
        seconds_value = _finite_number(seconds)
        if seconds_value is None or seconds_value < 0:
            raise ValueError("seconds は有限の非負数または None が必要です")
    return {
        "label": label,
        "status": "complete" if rows else "pending",
        **_group_summary(rows),
        "seconds": seconds,
        "by_grounding": {
            "grounded": _group_summary([row for row in rows if not row["gold_needs_review"]]),
            "ungrounded": _group_summary([row for row in rows if row["gold_needs_review"]]),
        },
    }


def _index_rows(rows: list[dict[str, Any]]) -> dict[str | int, dict[str, Any]]:
    _validate_rows(rows)
    indexed = {}
    for row in rows:
        row_id = row.get("id")
        if type(row_id) not in (str, int) or row_id == "":
            raise ValueError("比較する各 row に空でない文字列または整数の id が必要です")
        if row_id in indexed:
            raise ValueError(f"比較する id が重複しています: {row_id}")
        indexed[row_id] = row
    return indexed


def _metric_changes(before: dict, after: dict, key: str) -> dict[str, Any]:
    ids = list(before)
    improved = [row_id for row_id in ids if not before[row_id][key] and after[row_id][key]]
    regressed = [row_id for row_id in ids if before[row_id][key] and not after[row_id][key]]
    return {
        "improved_ids": improved,
        "regressed_ids": regressed,
        "persistent_failure_ids": [row_id for row_id in ids if not before[row_id][key] and not after[row_id][key]],
        "unchanged_success_ids": [row_id for row_id in ids if before[row_id][key] and after[row_id][key]],
        "remaining_failure_ids": [row_id for row_id in ids if not after[row_id][key]],
        "net_success_change": len(improved) - len(regressed),
        "rate_change": (len(improved) - len(regressed)) / len(ids) if ids else None,
    }


def compare_rows(before_rows: list[dict[str, Any]], after_rows: list[dict[str, Any]]) -> dict[str, Any]:
    """同一 id を対応させて比較。順序違いは許容し、集合違いは停止する。"""
    before = _index_rows(before_rows)
    after = _index_rows(after_rows)
    if before.keys() != after.keys():
        missing = [row_id for row_id in before if row_id not in after]
        extra = [row_id for row_id in after if row_id not in before]
        raise ValueError(f"学習前後の id 集合が異なります: after にない={missing}, after のみ={extra}")
    for row_id in before:
        if before[row_id]["gold_needs_review"] is not after[row_id]["gold_needs_review"]:
            raise ValueError(f"学習前後で正解ラベルが異なります: {row_id}")
        for key in ("question", "category"):
            if key in before[row_id] or key in after[row_id]:
                if before[row_id].get(key) != after[row_id].get(key):
                    raise ValueError(f"学習前後で {key} が異なります: {row_id}")
    by_metric = {
        name: _metric_changes(before, after, key)
        for name, key in (("format", "format_ok"), ("content", "content_ok"), ("both", "both_ok"))
    }
    return {
        "n": len(before),
        "status": "complete" if before else "pending",
        **by_metric["both"],
        "by_metric": by_metric,
    }


def _cell(value: Any) -> str:
    if value is None:
        return "未記録"
    return (
        str(value).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        .replace("|", "&#124;").replace("\r", "").replace("\n", "<br>")
    )


def _number(value: Any, digits: int = 3) -> str:
    number = _finite_number(value)
    return f"{number:,.{digits}f}" if number is not None else "未記録"


def _rate(value: Any) -> str:
    number = _finite_number(value)
    return f"{number * 100:.1f}%" if number is not None else "未測定"


def _rate_count(summary: Mapping[str, Any], key: str) -> str:
    if not summary["n"]:
        return "未測定（0件）"
    count_key = {"format_rate": "format_ok_count", "content_rate": "content_ok_count", "both_rate": "both_ok_count"}[key]
    return f"{_rate(summary[key])}（{summary[count_key]}/{summary['n']}）"


def _as_text(value: Any) -> str:
    if value is None:
        return "未記録"
    if isinstance(value, Mapping):
        return " / ".join(f"{key}: {item}" for key, item in value.items())
    if isinstance(value, (list, tuple)):
        return " / ".join(str(item) for item in value)
    return str(value)


def _ids(ids: list[Any]) -> str:
    return "、".join(_cell(value) for value in ids) if ids else "なし"


def _metadata_value(metadata: Mapping[str, Any], *names: str) -> Any:
    for name in names:
        if name in metadata:
            return metadata[name]
    return None


def _split_count(counts: Mapping[str, Any], name: str) -> Any:
    value = counts.get(name)
    return value.get("total") if isinstance(value, Mapping) else value


def _delta_sentence(before: Mapping[str, Any], after: Mapping[str, Any], key: str, label: str) -> str:
    if not before["n"] or not after["n"]:
        return f"{label}の変化は未測定。"
    delta = after[key] - before[key]
    word = "上昇" if delta > 0 else "低下" if delta < 0 else "変化なし"
    return f"{label}は {_rate(before[key])} → {_rate(after[key])}、{abs(delta) * 100:.1f} ポイントの{word}。"


def _summary_matches(recorded: Mapping[str, Any] | None, actual: Mapping[str, Any], name: str) -> None:
    if not recorded:
        return
    if not isinstance(recorded, Mapping):
        raise ValueError(f"result.{name} は辞書が必要です")
    for key in ("n", "format_rate", "content_rate", "both_rate"):
        if key not in recorded:
            continue
        left, right = recorded[key], actual[key]
        if left is None or right is None:
            if left is not right:
                raise ValueError(f"result.{name}.{key} が行からの集計と異なります")
        elif type(left) not in (int, float) or not math.isfinite(left) or not math.isclose(left, right, rel_tol=1e-9, abs_tol=1e-9):
            raise ValueError(f"result.{name}.{key} が行からの集計と異なります")


def render_report(
    result: Mapping[str, Any],
    before_rows: list[dict[str, Any]],
    after_rows: list[dict[str, Any]],
    metadata: Mapping[str, Any],
) -> str:
    """実測値から日本語 Markdown 下書きを作る。API・学習は実行しない。

    実行情報の欠落は未完了と表示する。既存の集計と行データの不一致は停止する。
    """
    config = result.get("config") or {}
    versions = result.get("versions") or {}
    training = result.get("training") or {}
    recorded_before = result.get("before") or {}
    recorded_after = result.get("after") or {}
    before = summarize_rows(before_rows, "before", recorded_before.get("seconds"))
    after = summarize_rows(after_rows, "after", recorded_after.get("seconds"))
    _summary_matches(recorded_before, before, "before")
    _summary_matches(recorded_after, after, "after")
    comparison = compare_rows(before_rows, after_rows) if before_rows and after_rows else None
    missing = []
    if result.get("executed_training_myself") is not True:
        missing.append("本人による学習実行の確認")
    if not before_rows:
        missing.append("学習前の評価")
    if not after_rows:
        missing.append("学習後の評価")
    for key, label in (("seconds", "学習時間"), ("steps", "学習ステップ数"), ("epochs", "学習エポック数"), ("final_logged_loss", "最終記録loss")):
        number = _finite_number(training.get(key))
        if number is None or number < 0 or (key in ("steps", "epochs") and number == 0):
            missing.append(label)
    knowledge = result.get("knowledge") or []
    if not knowledge:
        missing.append("追加の知識質問の評価")
    title = _metadata_value(metadata, "title", "topic")
    lines = ["# 第2回 解答（実測ログから作成した下書き）", ""]
    if missing:
        lines += [
            "**未完了・未実測の項目があります。学習と比較の完了を示すレポートとしては未確定です。**",
            "", "未確認の項目：" + "、".join(missing) + "。", "",
        ]
    else:
        lines += ["本人による学習と学習前後の評価ログから、観測された値を以下に記録します。", ""]
    lines += [
        "## 演習", "", "### 実行した内容と設定", "",
        f"- 題材：{_cell(title)}",
        f"- 架空の題材か：{'はい' if metadata.get('fictional') is True else '未記録' if metadata.get('fictional') is None else 'いいえ'}",
        f"- 本人が学習を実行したか：{'はい' if result.get('executed_training_myself') is True else 'いいえ' if result.get('executed_training_myself') is False else '未確認'}",
        f"- 実行ID：{_cell(result.get('run_id'))}",
        f"- 開始日時：{_cell(result.get('started_at'))}",
        f"- 完了日時：{_cell(result.get('completed_at'))}",
        f"- 学習方式：{_cell(result.get('method'))}",
        f"- 参考結果の出典：{_cell(metadata.get('reference_source'))}" if metadata.get("reference_source") else "- 結果の由来：この実行で保存された評価行・学習ログ",
        "", "| 設定 | 値 |", "|---|---|",
    ]
    for key in ("base_model", "revision", "seed", "num_train_epochs", "lr", "lora_r", "lora_alpha", "max_seq_len", "max_new_tokens", "gen"):
        lines.append(f"| {key} | {_cell(_as_text(config.get(key)))} |")
    lines += ["", "ライブラリの版：" + _cell(_as_text(versions or None)) + "。", ""]
    counts = metadata.get("split_counts") or {}
    lines += [
        "### データ", "",
        f"- train：{_cell(_split_count(counts, 'train'))}件 / valid：{_cell(_split_count(counts, 'valid'))}件 / test：{_cell(_split_count(counts, 'test'))}件",
        "- これは独自FAQを用いた実装課題の実行記録です。",
    ]
    faq = _metadata_value(metadata, "faq_items", "faq", "FAQ")
    if isinstance(faq, Mapping):
        for key, text in faq.items():
            lines.append(f"- {_cell(key)}：{_cell(_as_text(text))}")
    elif isinstance(faq, list):
        for item in faq:
            lines.append("- " + _cell(_as_text(item)))
    lines += [
        "", "### 学習の記録", "", "| 項目 | 実測・記録値 |", "|---|---|",
        f"| 学習時間（秒） | {_number(training.get('seconds'))} |",
        f"| ステップ数 | {_number(training.get('steps'), 0)} |",
        f"| 実行エポック数 | {_number(training.get('epochs'), 3)} |",
        f"| 最後に記録されたloss | {_number(training.get('final_logged_loss'), 6)} |",
        f"| 学習全体の平均loss | {_number(training.get('mean_training_loss'), 6)} |",
        f"| 学習対象パラメータ数 | {_number(training.get('trainable_parameters'), 0)} |",
        f"| 全パラメータ数 | {_number(training.get('total_parameters'), 0)} |",
        "",
        "最終記録lossと学習全体の平均lossは別の値として扱う。欠けた値は0に補完しない。",
        "", "### 学習前後の比較", "",
        "| 条件 | 評価件数 | 形式遵守率 | 内容正解率 | 形式・内容とも一致 | 評価時間（秒） |",
        "|---|---:|---|---|---|---:|",
    ]
    for label, summary in (("学習前", before), ("学習後", after)):
        lines.append(f"| {label} | {summary['n']} | {_rate_count(summary, 'format_rate')} | {_rate_count(summary, 'content_rate')} | {_rate_count(summary, 'both_rate')} | {_number(summary['seconds'])} |")
    lines += [
        "", _delta_sentence(before, after, "format_rate", "形式遵守率"),
        "", _delta_sentence(before, after, "content_rate", "内容正解率"),
        "",
        "**内容正解率は、needs_reviewの真偽値と必須語句の一致に基づく指標であり、回答の意味を完全に判定する指標ではない。** 説明文やコードフェンスが付いた回答は形式違反とする。一つのJSONオブジェクトを抽出できた場合は内容を独立に採点する。複数のJSON回答から都合のよいものを選ばない。",
        "",
        "| 質問区分 | 条件 | 件数 | 形式遵守率 | 内容正解率 | 両方一致 |",
        "|---|---|---:|---|---|---|",
    ]
    for group, group_label in (("grounded", "根拠あり"), ("ungrounded", "根拠なし")):
        for label, summary in (("学習前", before), ("学習後", after)):
            sub = summary["by_grounding"][group]
            lines.append(f"| {group_label} | {label} | {sub['n']} | {_rate_count(sub, 'format_rate')} | {_rate_count(sub, 'content_rate')} | {_rate_count(sub, 'both_rate')} |")
    lines += ["", "### 改善・退行・残った失敗", ""]
    if comparison is None:
        lines += ["未測定。学習前後の同一質問IDの評価がそろってから比較する。", ""]
    else:
        lines += [
            "以下は形式・内容の両方が一致したかを基準に、質問IDで対応付けた結果である。", "",
            "- 改善したID：" + _ids(comparison["improved_ids"]),
            "- 退行したID：" + _ids(comparison["regressed_ids"]),
            "- 前後とも失敗したID：" + _ids(comparison["persistent_failure_ids"]),
            "- 学習後に失敗が残ったID：" + _ids(comparison["remaining_failure_ids"]), "",
        ]
    lines += ["| 主な失敗分類 | 学習前 | 学習後 |", "|---|---:|---:|"]
    for kind in FAILURE_TYPES[1:]:
        b_count = before["failure_type_counts"][kind] if before["n"] else "未測定"
        a_count = after["failure_type_counts"][kind] if after["n"] else "未測定"
        lines.append(f"| {FAILURE_LABELS[kind]} | {b_count} | {a_count} |")
    lines += ["", "分類では形式違反を優先する。形式違反と内容不一致が同時に起きた場合の主分類は形式の崩れとする。", ""]
    failures = [row for row in after_rows if not row["both_ok"]]
    if failures:
        lines += ["| ID | 質問 | 学習後の失敗分類 | 学習後の出力原文 |", "|---|---|---|---|"]
        for row in failures:
            lines.append(f"| {_cell(row.get('id'))} | {_cell(row.get('question'))} | {FAILURE_LABELS[row['failure_type']]} | {_cell(row.get('raw'))} |")
        lines += [""]
    elif after_rows:
        lines += ["今回の評価集合では、学習後に形式・内容指標の失敗は残らなかった。", ""]
    else:
        lines += ["学習後の残存失敗は未測定。", ""]
    lines += ["### 知識を問う追加質問", ""]
    if knowledge:
        lines += ["| ID | 質問 | FAQに根拠があるか | 学習前の分類 | 学習後の分類 | 学習後の原文 |", "|---|---|---|---|---|---|"]
        for row in knowledge:
            b, a = row.get("before") or {}, row.get("after") or {}
            grounded = "あり" if row.get("grounded") is True else "なし" if row.get("grounded") is False else "未記録"
            lines.append(f"| {_cell(row.get('id'))} | {_cell(row.get('question'))} | {grounded} | {_cell(b.get('classification'))} | {_cell(a.get('classification'))} | {_cell(a.get('raw'))} |")
        lines += ["", "根拠なしの質問でassertedとなった回答は、FAQに支えられない断定として確認する。根拠ありの対照質問のassertedとは分けて読む。", ""]
    else:
        lines += ["未実測。追加質問の結果はまだ記録されていない。", ""]
    lines += [
        "## 宿題", "",
        "- 取り組んだレベル：実装",
        "- 変更した内容：独自の題材にFAQ・根拠あり質問・根拠なし質問を置き換え、同一の出力契約で学習前後を比較する設計。",
        "- 根拠なし質問の用意方法：" + _cell(_as_text(metadata.get("ungrounded_question_design"))),
        "- 追加質問の設計：" + _cell(_as_text(metadata.get("probe_design"))),
        "- 採点設計の注記：" + _cell(_as_text(metadata.get("scoring_notes"))),
        "", "### 観測された変化", "",
        _delta_sentence(before, after, "both_rate", "形式・内容の両方が一致した割合"), "",
    ]
    if comparison is not None:
        lines += [
            f"両方一致へ改善した質問は{len(comparison['improved_ids'])}件、退行した質問は{len(comparison['regressed_ids'])}件、学習後に失敗が残った質問は{len(comparison['remaining_failure_ids'])}件だった。", "",
        ]
    else:
        lines += ["学習が改善・悪化・変化なしのどれに当たるかは、実測がそろうまで判断しない。", ""]
    lines += [
        "### 考察を仕上げるときの論点", "",
        "- 形式遵守率と内容正解率の変化を別々に説明する。lossの低下だけから回答品質の改善を結論にしない。",
        "- 根拠なし質問への断定、根拠あり質問への不要な確認要求、形式違反を原文と結び付けて説明する。",
        "- 同じFAQに関する未学習の言い換えを評価している。未知の制度や新しい事実まで学習した証拠としては扱わない。",
        "- 今回の題材で回答形式をそろえる学習の効果を検討し、頻繁に変わる情報を外部参照で与える設計との役割分担を考察する。",
        "",
    ]
    return "\n".join(lines).rstrip() + "\n"

## 4. 学習と結果保存の手順

教材と同じモデル、revision、LoRA設定、学習量、貪欲生成を使います。
量子化なしの基盤モデルで、同じ質問・同じ生成条件を学習前後に評価します。
学習前の結果を先に保存し、学習時間、損失、質問ごとの出力、残った失敗を記録します。

次の定義セルのあとにある「実行」セルが、モデルのダウンロードと学習を開始します。

In [ ]:
%%writefile /content/day2_lab_code/day2_runtime.py
"""Run one Day2 LoRA experiment in a fresh Colab GPU work directory.

Importing this module does not import the training stack or start a run.
The model and training settings follow the course notebook. All evaluation
outputs are saved before training, and every generated row is saved as it is
obtained so that an interrupted run remains inspectable.
"""

from __future__ import annotations

import copy
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import math
import numbers
from pathlib import Path
import platform
import subprocess
import sys
import time
import traceback
from typing import Any, Callable, Mapping
import uuid


PINS = {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}

DEFAULT_CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},
}

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]


def _now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def _json_value(value: Any) -> Any:
    """Keep JSON valid; a non-finite measurement is missing, never zero."""
    if value is None or isinstance(value, (str, bool)):
        return value
    if isinstance(value, numbers.Integral):
        return int(value)
    if isinstance(value, numbers.Real):
        return float(value) if math.isfinite(float(value)) else None
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, Mapping):
        return {str(k): _json_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_value(v) for v in value]
    raise TypeError(f"JSON に保存できない型です: {type(value).__name__}")


def _write_json(path: Path, value: Any) -> None:
    payload = json.dumps(_json_value(value), ensure_ascii=False,
                         indent=2, allow_nan=False) + "\n"
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(payload, encoding="utf-8")
    temp.replace(path)


def _write_jsonl(path: Path, rows: list[dict]) -> None:
    payload = "".join(json.dumps(_json_value(row), ensure_ascii=False,
                                 allow_nan=False) + "\n" for row in rows)
    path.write_text(payload, encoding="utf-8")


def _configuration(config: Mapping[str, Any]) -> dict:
    resolved = copy.deepcopy(DEFAULT_CONFIG)
    unknown = set(config) - set(DEFAULT_CONFIG)
    if unknown:
        raise ValueError(f"未対応の設定項目: {sorted(unknown)}")
    resolved.update(copy.deepcopy(dict(config)))
    # This runner implements one controlled run of the implementation task.
    if resolved != DEFAULT_CONFIG:
        changed = [k for k in DEFAULT_CONFIG if resolved[k] != DEFAULT_CONFIG[k]]
        raise ValueError("今回は講座の標準設定を固定します。異なる項目: "
                         + ", ".join(changed))
    return resolved


def _input_messages(example: Mapping[str, Any], build_messages: Callable) -> list:
    # Withhold every target/label field from the inference message builder.
    messages = build_messages({"faq": example["faq"],
                               "question": example["question"]})
    if not isinstance(messages, list) or [m.get("role") for m in messages] != ["system", "user"]:
        raise ValueError("推論入力は system と user の2メッセージにしてください。")
    return messages


def _validate_lengths(tokenizer: Any, dataset: Mapping[str, Any],
                      build_messages: Callable, to_chat: Callable,
                      max_seq_len: int) -> dict:
    """Validate untruncated teacher sequences and generation prompts."""
    rows = []
    for split in ("train", "valid"):
        for example in dataset[split]:
            chat = to_chat(example)
            messages = chat["messages"]
            expected = _input_messages(example, build_messages) + [
                {"role": "assistant", "content": example["target"]}
            ]
            if messages != expected:
                raise ValueError(f"教師メッセージが推論入力＋正解と一致しません: {example['id']}")
            # SFTTrainer receives these same chat messages. Check both the
            # direct chat tokenization and rendered-text special-token case.
            direct = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=False)
            rendered = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False)
            rendered_ids = tokenizer(rendered, truncation=False)["input_ids"]
            count = max(len(direct), len(rendered_ids))
            rows.append({"id": example["id"], "split": split,
                         "teacher_included": True, "tokens": count,
                         "chat_template_tokens": len(direct),
                         "rendered_text_tokens": len(rendered_ids)})
    for split in ("test", "probes"):
        for example in dataset[split]:
            prompt = tokenizer.apply_chat_template(
                _input_messages(example, build_messages), tokenize=False,
                add_generation_prompt=True, enable_thinking=False)
            count = len(tokenizer(prompt, truncation=False)["input_ids"])
            rows.append({"id": example["id"], "split": split,
                         "teacher_included": False, "tokens": count})
    too_long = [row for row in rows if row["tokens"] > max_seq_len]
    return {"max_seq_len": max_seq_len, "checked_sequences": len(rows),
            "max_observed_tokens": max(row["tokens"] for row in rows),
            "truncation_requested": False, "passed": not too_long,
            "too_long": too_long, "rows": rows}


def _versions() -> dict:
    names = list(PINS) + ["torch", "tokenizers", "huggingface-hub",
                          "safetensors", "numpy", "pandas", "pyarrow"]
    versions = {"python": platform.python_version()}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def _gpu_environment(torch: Any) -> dict:
    try:
        info = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"], capture_output=True, text=True,
            timeout=10, check=False).stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        info = None
    device = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device)
    return {"platform": platform.platform(), "device": "cuda",
            "gpu_name": torch.cuda.get_device_name(device),
            "gpu_total_memory_bytes": int(props.total_memory),
            "gpu_compute_capability": list(torch.cuda.get_device_capability(device)),
            "torch_cuda_version": torch.version.cuda,
            "cudnn_version": torch.backends.cudnn.version(),
            "nvidia_smi": info, "model_dtype": "float32",
            "quantization": None, "fp16": False, "bf16": False}


def _classification(score: Mapping[str, Any], raw: str) -> str:
    if not score["format_ok"]:
        return "format_error"
    obj = json.loads(raw)
    return "needs_review" if obj["needs_review"] else "asserted"


def _generate(model: Any, tokenizer: Any, torch: Any, example: dict,
              config: dict, build_messages: Callable) -> tuple[str, dict]:
    prompt = tokenizer.apply_chat_template(
        _input_messages(example, build_messages), tokenize=False,
        add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=False).to(model.device)
    input_tokens = int(inputs["input_ids"].shape[1])
    if input_tokens > config["max_seq_len"]:
        raise ValueError(f"入力が長さ上限を超えています: {example['id']} ({input_tokens})")
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(**inputs,
                                max_new_tokens=config["max_new_tokens"],
                                pad_token_id=tokenizer.pad_token_id,
                                **config["gen"])
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    ids = output[0][input_tokens:].tolist()
    raw = tokenizer.decode(ids, skip_special_tokens=True).strip()
    eos = getattr(model.generation_config, "eos_token_id", None)
    if eos is None:
        eos = tokenizer.eos_token_id
    eos_ids = list(eos) if isinstance(eos, (list, tuple)) else [eos]
    ended_with_eos = bool(ids) and ids[-1] in eos_ids
    hit_limit = len(ids) >= config["max_new_tokens"]
    diagnostics = {
        "input_tokens": input_tokens, "generated_tokens": len(ids),
        "token_count_includes_special_tokens": True,
        "max_new_tokens": config["max_new_tokens"],
        "hit_max_new_tokens": hit_limit, "ended_with_eos": ended_with_eos,
        "possible_truncation": hit_limit and not ended_with_eos,
        "last_token_id": ids[-1] if ids else None,
        "stop_reason": "eos" if ended_with_eos else ("max_new_tokens" if hit_limit else "other"),
        "seconds": round(elapsed, 6),
    }
    return raw, diagnostics


def _evaluate(model: Any, tokenizer: Any, torch: Any, examples: list[dict],
              gold_for: Callable, config: dict, label: str, path: Path,
              build_messages: Callable, score_output: Callable,
              summarize_rows: Callable) -> tuple[dict, list[dict]]:
    model.eval()
    rows = []
    start = time.perf_counter()
    _write_json(path, {"label": label, "completed": False, "summary": None, "rows": rows})
    for index, example in enumerate(examples, 1):
        gold = gold_for(example)
        if type(gold["needs_review"]) is not bool:
            raise TypeError(f"正解 needs_review の型が真偽値ではありません: {example['id']}")
        raw, generation = _generate(model, tokenizer, torch, example, config, build_messages)
        score = score_output(raw, gold)
        row = {**score, "id": example["id"], "question": example["question"],
               "category": example["category"], "raw": raw,
               "gold_needs_review": gold["needs_review"],
               "gold": gold, "generation": generation,
               "classification": _classification(score, raw)}
        rows.append(row)
        _write_json(path, {"label": label, "completed": False,
                           "summary": None, "rows": rows})
        print(f"{label}: {index}/{len(examples)}", flush=True)
    summary = summarize_rows(rows, label, time.perf_counter() - start)
    _write_json(path, {"label": label, "completed": True, "summary": summary, "rows": rows})
    print(json.dumps(_json_value(summary), ensure_ascii=False), flush=True)
    return summary, rows


def run_experiment(config: Mapping[str, Any], work_dir: str | Path) -> Path:
    """Train once on a CUDA GPU and return the newly created run directory."""
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    work = Path(work_dir).expanduser() / run_id
    work.mkdir(parents=True, exist_ok=False)
    started_at = _now()
    stage = "configuration"
    training_completed = False
    comparison_completed = False
    state = {"run_id": run_id, "started_at": started_at,
             "status": "running", "executed_training_myself": False}
    _write_json(work / "run_status.json", state)
    print("今回の保存先:", work, flush=True)
    try:
        cfg = _configuration(config)
        _write_json(work / "config.json", cfg)
        stage = "environment"
        versions = _versions()
        _write_json(work / "versions.json", versions)
        mismatch = {name: {"expected": expected, "actual": versions[name]}
                    for name, expected in PINS.items() if versions[name] != expected}
        if mismatch:
            raise RuntimeError("講座の固定版と異なります。導入セルを実行してセッションを再起動してください: "
                               + json.dumps(mismatch, ensure_ascii=False))
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA GPU が見つかりません。Colab の「ランタイム」→"
                               "「ランタイムのタイプを変更」で T4 GPU を選んでください。")
        environment = _gpu_environment(torch)
        _write_json(work / "environment.json", environment)
        packages = {dist.metadata.get("Name", "unknown"): dist.version
                    for dist in importlib.metadata.distributions()}
        _write_json(work / "installed_packages.json", dict(sorted(packages.items())))

        from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
        from peft import LoraConfig
        from trl import SFTTrainer, SFTConfig
        from datasets import Dataset
        import day2_data
        import day2_evaluation

        stage = "dataset"
        dataset = day2_data.build_dataset()
        counts = {name: len(dataset[name]) for name in ("train", "valid", "test", "probes")}
        if counts != {"train": 14, "valid": 7, "test": 21, "probes": 4}:
            raise ValueError(f"今回の学習・評価件数が想定と異なります: {counts}")
        for split in ("train", "valid", "test", "probes"):
            _write_jsonl(work / f"{split}.jsonl", dataset[split])
        _write_json(work / "gold.json", dataset["gold"])
        metadata = copy.deepcopy(dataset["metadata"])
        canonical = json.dumps({key: dataset[key] for key in ("train", "valid", "test", "gold", "probes")},
                               ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        metadata["dataset_sha256"] = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
        metadata["observed_split_counts"] = counts
        _write_json(work / "dataset_metadata.json", metadata)
        source_hashes = {}
        for name, module in (("day2_runtime", sys.modules[__name__]),
                             ("day2_data", day2_data), ("day2_evaluation", day2_evaluation)):
            source = Path(module.__file__)
            source_hashes[name] = hashlib.sha256(source.read_bytes()).hexdigest()
        _write_json(work / "source_sha256.json", source_hashes)

        stage = "tokenizer_and_length_check"
        set_seed(cfg["seed"])
        tokenizer = AutoTokenizer.from_pretrained(cfg["base_model"], revision=cfg["revision"])
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        lengths = _validate_lengths(tokenizer, dataset, day2_data.build_messages,
                                    day2_data.to_chat, cfg["max_seq_len"])
        _write_json(work / "sequence_lengths.json", lengths)
        if not lengths["passed"]:
            raise ValueError("教師回答を含む長さ、または評価入力が512トークンを超えました。"
                             "切り詰めずに停止します。sequence_lengths.json を確認してください。")

        stage = "model_loading"
        model = AutoModelForCausalLM.from_pretrained(
            cfg["base_model"], revision=cfg["revision"], dtype=torch.float32).to("cuda")
        model.eval()
        model.config.use_cache = True
        base_memory_bytes = int(model.get_memory_footprint())
        print("モデル:", cfg["base_model"], "| GPU:", environment["gpu_name"],
              "| 重み GB:", round(base_memory_bytes / 1e9, 3), flush=True)

        evaluation_args = {"tokenizer": tokenizer, "torch": torch, "config": cfg,
                           "build_messages": day2_data.build_messages,
                           "score_output": day2_evaluation.score_output,
                           "summarize_rows": day2_evaluation.summarize_rows}
        stage = "evaluation_before"
        before_summary, before_rows = _evaluate(
            model=model, examples=dataset["test"],
            gold_for=lambda example: dataset["gold"][example["question"]],
            label="before", path=work / "eval_before.json", **evaluation_args)
        stage = "probes_before"
        _, probes_before = _evaluate(
            model=model, examples=dataset["probes"], gold_for=lambda example: example["gold"],
            label="probes_before", path=work / "probes_before.json", **evaluation_args)

        stage = "training_setup"
        train_ds = Dataset.from_list([day2_data.to_chat(example) for example in dataset["train"]])
        valid_ds = Dataset.from_list([day2_data.to_chat(example) for example in dataset["valid"]])
        model.config.use_cache = False
        lora = LoraConfig(r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
                          lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                          target_modules=LORA_TARGET_MODULES)
        sft = SFTConfig(
            output_dir=str(work / "ckpt"), num_train_epochs=cfg["num_train_epochs"],
            per_device_train_batch_size=4, gradient_accumulation_steps=2,
            learning_rate=cfg["lr"], lr_scheduler_type="cosine",
            warmup_steps=cfg["warmup_steps"], logging_steps=10,
            save_steps=10, save_total_limit=2, fp16=False, bf16=False,
            max_length=cfg["max_seq_len"], packing=False, report_to="none", seed=cfg["seed"])
        trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds,
                             eval_dataset=valid_ds, peft_config=lora, processing_class=tokenizer)
        trainable_parameters = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
        total_parameters = sum(p.numel() for p in trainer.model.parameters())
        trainer.model.print_trainable_parameters()
        stage = "training"
        torch.cuda.synchronize()
        training_start = time.perf_counter()
        train_output = trainer.train()
        torch.cuda.synchronize()
        training_seconds = time.perf_counter() - training_start
        loss_history = [dict(row) for row in trainer.state.log_history if "loss" in row]
        final_logged_loss = loss_history[-1]["loss"] if loss_history else None
        mean_training_loss = train_output.metrics.get("train_loss", train_output.training_loss)
        training = {
            "seconds": round(training_seconds, 6), "steps": int(trainer.state.global_step),
            "epochs": trainer.state.epoch, "final_logged_loss": final_logged_loss,
            "mean_training_loss": mean_training_loss, "loss_history": loss_history,
            "trainable_parameters": trainable_parameters, "total_parameters": total_parameters,
            "trainable_fraction": trainable_parameters / total_parameters,
            "trainer_metrics": dict(train_output.metrics),
            "seconds_definition": "trainer.train() の前後をCUDA同期して計測。途中チェックポイント保存を含む。",
            "final_logged_loss_definition": "学習ログの最後の loss。ログ区間の平均で、全学習の平均 train_loss と区別。",
        }
        _write_json(work / "training.json", training)
        _write_json(work / "trainer_log_history.json", trainer.state.log_history)
        if (final_logged_loss is None or not math.isfinite(float(final_logged_loss))
                or not math.isfinite(float(mean_training_loss))):
            raise RuntimeError("学習損失を有限の数値として確認できません。training.json を確認してください。")
        if training["steps"] != 40 or not math.isclose(float(training["epochs"]), 20.0):
            raise RuntimeError("20エポック・40ステップの学習が完了していません。training.json を確認してください。")
        training_completed = True
        stage = "adapter_save"
        trainer.model.save_pretrained(work / "adapter")
        tokenizer.save_pretrained(work / "adapter")
        tuned = trainer.model
        tuned.config.use_cache = True
        tuned.eval()

        stage = "evaluation_after"
        after_summary, after_rows = _evaluate(
            model=tuned, examples=dataset["test"],
            gold_for=lambda example: dataset["gold"][example["question"]],
            label="after", path=work / "eval_after.json", **evaluation_args)
        stage = "probes_after"
        _, probes_after = _evaluate(
            model=tuned, examples=dataset["probes"], gold_for=lambda example: example["gold"],
            label="probes_after", path=work / "probes_after.json", **evaluation_args)

        stage = "comparison"
        comparison = day2_evaluation.compare_rows(before_rows, after_rows)
        _write_json(work / "comparison.json", comparison)
        before_by_id = {row["id"]: row for row in probes_before}
        knowledge = []
        for after_row in probes_after:
            before_row = before_by_id[after_row["id"]]
            knowledge.append({
                "id": after_row["id"], "question": after_row["question"],
                "category": after_row["category"], "grounded": not after_row["gold_needs_review"],
                "gold": after_row["gold"],
                "before": {"raw": before_row["raw"], "classification": before_row["classification"],
                           "score": {k: v for k, v in before_row.items() if k not in
                                     ("id", "question", "category", "raw", "gold", "generation", "classification")},
                           "generation": before_row["generation"]},
                "after": {"raw": after_row["raw"], "classification": after_row["classification"],
                          "score": {k: v for k, v in after_row.items() if k not in
                                    ("id", "question", "category", "raw", "gold", "generation", "classification")},
                          "generation": after_row["generation"]},
            })
        _write_json(work / "knowledge.json", knowledge)
        comparison_completed = True
        result = {
            "config": cfg, "versions": versions, "method": "LoRA（量子化なし、float32）",
            "before": before_summary, "after": after_summary, "comparison": comparison,
            "training": training, "knowledge": knowledge, "executed_training_myself": True,
            "run_id": run_id, "started_at": started_at, "completed_at": _now(),
            "environment": environment, "base_memory_bytes": base_memory_bytes,
            "dataset_metadata": metadata, "sequence_length_check": {
                k: v for k, v in lengths.items() if k not in ("rows", "too_long")},
            "source_sha256": source_hashes, "status": "completed",
        }
        stage = "report"
        report = day2_evaluation.render_report(result, before_rows, after_rows, metadata)
        (work / "answer_generated.md").write_text(report, encoding="utf-8")
        _write_json(work / "result.json", result)
        state.update({"status": "completed", "completed_at": result["completed_at"],
                      "executed_training_myself": True})
        _write_json(work / "run_status.json", state)
        print("学習・学習前後の比較が完了しました。", flush=True)
        print("結果の保存先:", work, flush=True)
        return work
    except BaseException as error:
        failure = {
            "run_id": run_id, "status": "failed", "stage": stage,
            "started_at": started_at, "failed_at": _now(),
            "exception_type": type(error).__name__, "exception": str(error),
            "traceback": traceback.format_exc(),
            "training_completed": training_completed,
            "comparison_completed": comparison_completed,
            "executed_training_myself": training_completed and comparison_completed,
            "saved_files": sorted(p.name for p in work.iterdir() if p.is_file()),
        }
        _write_json(work / "failure.json", failure)
        state.update({"status": "failed", "stage": stage, "failed_at": failure["failed_at"],
                      "executed_training_myself": failure["executed_training_myself"]})
        _write_json(work / "run_status.json", state)
        print("実行を停止しました。取得済みの結果と failure.json は次に保存されています:", work,
              flush=True)
        raise

## 5. 実行：学習前 → 学習 → 学習後

進捗が表示されます。最後に「結果保存完了」と出たら、次のセルで結果をダウンロードします。
エラーで止まったときは、表示されたエラー内容を確認してください。

In [ ]:
RESULT_DIR = None
from copy import deepcopy
import importlib
import json

import day2_data
import day2_evaluation
import day2_runtime
for module in (day2_data, day2_evaluation, day2_runtime):
    importlib.reload(module)

CONFIG = deepcopy(day2_runtime.DEFAULT_CONFIG)
print("今回の設定:")
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
RESULT_DIR = day2_runtime.run_experiment(CONFIG, "/content/day2_dx_lab")
print("結果保存完了:", RESULT_DIR)

## 6. 結果を1つのZIPでダウンロード

`day2_results.zip` に設定、学習記録、学習前後の回答、採点結果、独自FAQとソースコードをまとめます。
ダウンロードが始まらない場合は、Colab左のファイル一覧から `/content/day2_results.zip` を選び、右クリックしてダウンロードできます。

**ダウンロードしたZIPを保存してください。**
実測結果を読んでから、GitHubに置く `lectures/02/answer.md` とフォームの説明文を仕上げます。

In [ ]:
from pathlib import Path
import json
import zipfile

if globals().get("RESULT_DIR") is None:
    raise RuntimeError("今回の実行が完了していません。実行セルのエラーを確認してください。")
RESULT_DIR = Path(RESULT_DIR)
required = ["result.json", "eval_before.json", "eval_after.json"]
missing = [name for name in required if not (RESULT_DIR / name).is_file()]
if missing:
    raise RuntimeError("まだ必要な結果がそろっていません: " + ", ".join(missing))
result_status = json.loads((RESULT_DIR / "result.json").read_text(encoding="utf-8"))
if result_status.get("status") != "completed" or result_status.get("executed_training_myself") is not True:
    raise RuntimeError("学習と比較の完了を確認できません。実行セルの出力を確認してください。")

SOURCE_METADATA = {'assignment': 'AIエンジニアリング実践講座2026 Day2', 'level': '実装', 'topic': '架空の製造業向けDX相談窓口FAQ', 'prepared_on': '2026-10-27', 'source_repository': 'https://github.com/matsuolab/lecture-ai-engineering-2026', 'source_commit': '662cddd52724b330f3f4cd15123d1e55cb474874', 'source_notebook': 'notebooks/lora_format_tuning.ipynb', 'source_notebook_sha256': '2edcfb6d1461dabdbfea8242a7ae4ec7c231f4d2e5cec61920a4317ce8694b00', 'module_sha256': {'day2_data.py': '0a6bd304a27fe652591bb30136bca8fb316467dff33dd62b750886b2b8739081', 'day2_evaluation.py': 'a90c0e12232a437c0e478d2b13ed57be68d6e8e099d05d94caa9b2d443e396aa', 'day2_runtime.py': '3a5c075506c23601ac7bdd6f79e45cb3f0e85d7d14aa45c22b1d680fd8361de5'}}
(RESULT_DIR / "source_metadata.json").write_text(
    json.dumps(SOURCE_METADATA, ensure_ascii=False, indent=2), encoding="utf-8"
)
ZIP_PATH = Path("/content/day2_results.zip")
excluded = {"adapter", "ckpt", "__pycache__"}
allowed_suffixes = {".json", ".jsonl", ".csv", ".md", ".txt", ".py"}
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULT_DIR.rglob("*")):
        relative = path.relative_to(RESULT_DIR)
        if path.is_file() and not excluded.intersection(relative.parts) and path.suffix in allowed_suffixes:
            archive.write(path, "day2_results/" + relative.as_posix())
    for path in sorted(MODULE_DIR.glob("day2_*.py")):
        archive.write(path, "day2_results/source/" + path.name)

print(f"ダウンロードするファイル: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size:,} bytes)")
print("結果ZIPを保存し、回答原文と集計を確認してください。")
from google.colab import files
files.download(str(ZIP_PATH))

## 実行後

1. ZIPのダウンロードと保存を確認します。
2. Colabの **ランタイム → ランタイムを接続解除して削除** を選びます。

### 再現情報と教材

- 基準となる講座ノートブック：`matsuolab/lecture-ai-engineering-2026` の `notebooks/lora_format_tuning.ipynb`
- 参照commit：`662cddd52724b330f3f4cd15123d1e55cb474874`
- 宿題レベル：実装。独自FAQ5項目で学習前後を比較。
- モデル：`Qwen/Qwen3.5-0.8B`。重みのrevisionは実行設定に固定。

保存されるレポートは、実測値を整理する下書きです。出力原文を確認してから提出用の文章を確定します。